In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini")
response = llm.invoke("Who is Yan LeCun? Tell me about his high achievement")

print(response.content)

You probably mean Yann LeCun. Briefly:

Who he is
- Yann LeCun is a French-born computer scientist and a leading figure in artificial intelligence and machine learning. He is a professor at New York University and was (for many years) the founding director of Facebook/Meta AI Research (FAIR).

His highest/most notable achievements
- Pioneer of convolutional neural networks (CNNs): LeCun invented and developed convolutional nets (notably the LeNet family, with LeNet-5) in the late 1980s–1990s for handwriting and document recognition. Those architectures and ideas are the foundation of almost every modern computer-vision system.
- Major role in the deep-learning revolution: His research on gradient-based learning and architectures for representation learning helped make deep neural networks practical and effective. His work, together with Geoffrey Hinton and Yoshua Bengio, underpins today’s advances in speech recognition, image recognition, autonomous vehicles, and many other AI applicat

In [3]:
from langchain_groq import ChatGroq
from pydantic import BaseModel

class Schema(BaseModel):
    price: float
    eps: float

llm = ChatOpenAI(model="gpt-4o")
response = llm.with_structured_output(Schema).invoke("Extract Price and EPS from this report: "
                      "NVIDIA reported quarterly EPS of 2.3 and "
                      "their current price is $100.")

print(response)

price=100.0 eps=2.3


In [4]:
from pydantic import BaseModel, Field

class File(BaseModel):
    path: str = Field(description="The path to the file to be created or modified")
    purpose: str = Field(description="The purpose of the file, e.g. 'main application logic', 'data processing module', etc.")

class Plan(BaseModel):
    name: str = Field(description="The name of app to be built")
    description: str = Field(description="A oneline description of the app to be built, e.g. 'A web application for managing personal finances'")
    techstack: str = Field(description="The tech stack to be used for the app, e.g. 'python', 'javascript', 'react', 'flask', etc.")
    features: list[str] = Field(description="A list of features that the app should have, e.g. 'user authentication', 'data visualization', etc.")
    files: list[File] = Field(description="A list of files to be created, each with a 'path' and 'purpose'")

user_prompt = "create a simple calculater web application"

prompt = f"""
You are the PLANNER agent. Convert the user prompt into a COMPLETE engineering project plan

User request: {user_prompt}

"""

response = llm.with_structured_output(Plan).invoke(prompt)

print(response)

name='Simple Calculator Web Application' description='A web application that functions as a basic calculator, supporting standard arithmetic operations.' techstack='HTML, CSS, JavaScript' features=['Basic arithmetic operations (addition, subtraction, multiplication, division)', 'Responsive design for mobile and desktop use', 'User-friendly interface', 'Error handling for invalid inputs'] files=[File(path='index.html', purpose='The main HTML file that structures the calculator layout.'), File(path='styles.css', purpose='The CSS file for styling the calculator, making it both functional and visually appealing.'), File(path='calculator.js', purpose='The JavaScript file responsible for handling the logic of arithmetic operations and user interactions.'), File(path='test.html', purpose='An HTML file for testing and debugging the functionality and responsiveness of the calculator.')]


In [5]:
from langgraph.constants import END
from langgraph.graph import StateGraph

def planner_prompt(user_prompt: str) -> str:
    prompt = f"""
You are the PLANNER agent. Convert the user prompt into a COMPLETE engineering project plan.

User request: {user_prompt}
    """
    return prompt

def planner_agent(state:dict) -> dict:
    users_prompt = state["user_prompt"]
    response = llm.with_structured_output(Plan).invoke(planner_prompt(user_prompt))
    if response is None:
        raise ValueError("Planner did not return a valid response.")
    return {"plan": response}

user_prompt = "create a simple calculater web application"

graph = StateGraph(dict)
graph.add_node("planner", planner_agent)
graph.set_entry_point("planner")

agent = graph.compile()

result = agent.invoke({"user_prompt": user_prompt})

print(result)

{'plan': Plan(name='Simple Calculator Web App', description='A web application that performs basic arithmetic operations such as addition, subtraction, multiplication, and division.', techstack='HTML, CSS, JavaScript', features=['User can perform addition of two numbers', 'User can perform subtraction of two numbers', 'User can perform multiplication of two numbers', 'User can perform division of two numbers', 'Responsive design for both desktop and mobile', 'Error handling for invalid inputs'], files=[File(path='public/index.html', purpose='Main HTML file to structure the calculator layout'), File(path='public/styles.css', purpose='Stylesheet to style the calculator and ensure responsiveness'), File(path='public/script.js', purpose='JavaScript file to control the calculator operations and handle events')])}


In [33]:
from pydantic import ConfigDict

class ImplementationTask(BaseModel):
    filepath: str = Field(description="The path to the file to be modified")
    task_description: str = Field(description="A detailed description of the task to be performed on the file, e.g. 'add user authentication', 'implement data processing logic', etc.")

#ConfigDict(extra="allow") is for able to add extra element
class TaskPlan(BaseModel):
    implementation_steps: list[ImplementationTask] = Field(description="A list of steps to be taken to implement the task")
    plan: Optional[str] = None
    #plan: dict = Field(default_factory=dict, description="Detailed plan data")
    #model_config = ConfigDict(extra="allow")

def architect_prompt(plan: str) -> str:
    prompt = f"""
You are the ARCHITECT agent. Given this project plan, break it down into explicit engineering tasks.

RULES:
- For each FILE in the plan, create one or more IMPLEMENTATION TASKS.
- In each task description:
    * Specify exactly what to implement.
    * Name the variables, functions, classes, and components to be defined.
    * Mention how this task depends on or will be used by previous tasks.
    * Include integration details: imports, expected function signatures, data flow.
- Order tasks so that dependencies are implemented first.
- Each step must be SELF-CONTAINED but also carry FORWARD the relevant context from earlier tasks.

Project Plan:
{plan}
    """
    return prompt

#Principle of Context Engineering tells that it is good for maintain the whole context
def architect_agent(state:dict) -> dict:
    plan: Plan = state["plan"]
    response = llm.with_structured_output(TaskPlan).invoke(architect_prompt(plan))
    if response is None:
        raise ValueError("Architect did not return a valid response.")
    response.plan = plan
    return {"task_plan": response}

user_prompt = "create a simple calculater web application"

graph = StateGraph(dict)
graph.add_node("planner", planner_agent)
graph.add_node("architect", architect_agent)
graph.add_edge(start_key="planner", end_key="architect")
graph.set_entry_point("planner")

agent = graph.compile()

result = agent.invoke({"user_prompt": user_prompt})
#result = agent.invoke({"user_prompt": user_prompt}, {"recursion_limit": 50})

print(result)

{'task_plan': TaskPlan(implementation_steps=[ImplementationTask(filepath='index.html', task_description='Create the complete calculator UI skeleton for the web app. Define the main container elements needed by the JavaScript logic in `script.js`: a top-level `.calculator` wrapper, a `.display` area containing a read-only `<input>` or `<div>` for showing the current expression/result, and a `.buttons` grid containing clickable controls for digits `0-9`, operators `+`, `-`, `*`, `/`, a decimal point `.`, an equals button `=`, and a clear/reset button `C`. Assign stable selectors/IDs/classes that `script.js` can query with `document.querySelector`/`document.getElementById` and that `styles.css` can target for layout and theming. Ensure the markup is structured for responsive centering and accessibility, including button labels and semantics suitable for keyboard-driven and click-driven interaction.'), ImplementationTask(filepath='styles.css', task_description='Implement the full visual de

In [34]:
import pathlib
import subprocess
from typing import Tuple

from langchain_core.tools import tool

PROJECT_ROOT = pathlib.Path.cwd() / "generated_project"


def safe_path_for_project(path: str) -> pathlib.Path:
    p = (PROJECT_ROOT / path).resolve()
    if PROJECT_ROOT.resolve() not in p.parents and PROJECT_ROOT.resolve() != p.parent and PROJECT_ROOT.resolve() != p:
        raise ValueError("Attempt to write outside project root")
    return p


@tool
def write_file(path: str, content: str) -> str:
    """Writes content to a file at the specified path within the project root."""
    p = safe_path_for_project(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        f.write(content)
    return f"WROTE:{p}"


@tool
def read_file(path: str) -> str:
    """Reads content from a file at the specified path within the project root."""
    p = safe_path_for_project(path)
    if not p.exists():
        return ""
    with open(p, "r", encoding="utf-8") as f:
        return f.read()


@tool
def get_current_directory() -> str:
    """Returns the current working directory."""
    return str(PROJECT_ROOT)


@tool
def list_files(directory: str = ".") -> str:
    """Lists all files in the specified directory within the project root."""
    p = safe_path_for_project(directory)
    if not p.is_dir():
        return f"ERROR: {p} is not a directory"
    files = [str(f.relative_to(PROJECT_ROOT)) for f in p.glob("**/*") if f.is_file()]
    return "\n".join(files) if files else "No files found."

@tool
def run_cmd(cmd: str, cwd: str = None, timeout: int = 30) -> Tuple[int, str, str]:
    """Runs a shell command in the specified directory and returns the result."""
    cwd_dir = safe_path_for_project(cwd) if cwd else PROJECT_ROOT
    res = subprocess.run(cmd, shell=True, cwd=str(cwd_dir), capture_output=True, text=True, timeout=timeout)
    return res.returncode, res.stdout, res.stderr


def init_project_root():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    return str(PROJECT_ROOT)

In [39]:
from typing import Optional
from langchain.agents import create_agent

class CoderState(BaseModel):
    task_plan: TaskPlan = Field(description="The plan for the task to be implemented")
    current_step_idx: int = Field(0, description="The index of the current step in the implementation steps")
    current_file_content: Optional[str] = Field(None, description="The content of the file currently being edited or created")
    status: Optional[str] = None

def coder_system_prompt() -> str:
    prompt = """
You are the CODER agent.
You are implementing a specific engineering task.
You have access to tools to read and write files.

Always:
- Review all existing files to maintain compatibility.
- Implement the FULL file content, integrating with other modules.
- Maintain consistent naming of variables, functions, and imports.
- When a module is imported from another file, ensure it exists and is implemented as described.
    """
    return prompt

def coder_agent(state:dict) -> dict:
    coder_state: CoderState = state.get("coder_state")
    #First coder_state is none, only receive task plan
    if coder_state is None:
        coder_state = CoderState(task_plan=state["task_plan"], current_step_idx=0)

    steps = coder_state.task_plan.implementation_steps
    if coder_state.current_step_idx >= len(steps):
        return {"coder_state": coder_state, "status": "DONE"}

    current_task = steps[coder_state.current_step_idx]
    existing_content = read_file.run(current_task.filepath)

    system_prompt = coder_system_prompt()
    user_prompt = (
        f"Task: {current_task.task_description}\n"
        f"File: {current_task.filepath}\n"
        f"Existing Conetent:\n{existing_content}\n"
        "Use write_file(path, content) to save your changes"
    )

    coder_tools = [read_file, write_file, list_files, get_current_directory]

    react_agent = create_agent(llm, coder_tools)
    react_agent.invoke({"messages": [{"role": "system", "content": system_prompt},
                                     {"role": "user", "content": user_prompt}]})

    coder_state.current_step_idx += 1

    return {"coder_state": coder_state}


In [40]:
CoderState.model_rebuild()

user_prompt = "create a simple calculater web application"

graph = StateGraph(dict)
graph.add_node("planner", planner_agent)
graph.add_node("architect", architect_agent)
graph.add_node("coder", coder_agent)
graph.add_edge(start_key="planner", end_key="architect")
graph.add_edge(start_key="architect", end_key="coder")

graph.add_conditional_edges(
    "coder",#Source node
    lambda s: "END" if s.get("status") == "DONE" else "coder",
    {"END":END, "coder":"coder"}#Path_map
)
graph.set_entry_point("planner")

agent = graph.compile()

result = agent.invoke({"user_prompt": user_prompt}, {"recursion_limit": 50})

print(result)

{'coder_state': CoderState(task_plan=TaskPlan(implementation_steps=[ImplementationTask(filepath='index.html', task_description='Create the full calculator markup for the app shell and wire up semantic hooks for JavaScript and CSS. Define a main container such as <main class="calculator"> containing a display region with an output element for the current expression/result (for example an <input id="display" type="text" readonly> or equivalent <div id="display">), and a button grid container with buttons for digits 0-9, decimal point, operators (+, -, *, /), equals (=), clear (C), and backspace (⌫). Assign stable ids/data attributes to each control so script.js can attach listeners without relying on text content alone, and make sure the layout supports keyboard focus. Include the script and stylesheet imports at the bottom/head respectively using <link rel="stylesheet" href="styles.css"> and <script src="script.js" defer></script>, so script.js can safely query DOM nodes after parsing. 

In [42]:
user_prompt = "create a simple clock web application"

result = agent.invoke({"user_prompt": user_prompt}, {"recursion_limit": 50})

print(result)

{'coder_state': CoderState(task_plan=TaskPlan(implementation_steps=[ImplementationTask(filepath='index.html', task_description='Create the base HTML document for the clock app: define the standard `<!doctype html>`, `<html>`, `<head>`, and `<body>` structure; set the page `<title>` to something like `Simple Clock`; include `<meta charset="UTF-8">` and responsive viewport meta; link the stylesheet with `<link rel="stylesheet" href="styles.css">` and load the script with `<script src="script.js" defer></script>` so the JavaScript runs after the DOM is available. In the `<body>`, add a centered clock container element such as `<main class="clock-app">` containing a clock display element with a stable hook for JavaScript, e.g. `<div id="clock" class="clock" aria-live="polite">00:00:00</div>`. This element will be the target for `script.js` to update every second, so the id/class names must remain consistent with the JavaScript task and the styling task.'), ImplementationTask(filepath='styl